In [ ]:
import json
from collections import Counter
from pathlib import Path
import pandas as pd
from typing import Any, List
import numpy as np

# Merge Json of each image to a single json file

In [ ]:
# EDIT THESE TWO PATHS
SRC_ROOT = Path("/Users/shravanthij/Documents/ML academy/driving-perception-validation/data/01_raw/bdd100k/raw_labels")  # the folder containing train/ val/ test/
OUT_DIR = Path("/Users/shravanthij/Documents/ML academy/driving-perception-validation/data/01_raw/bdd100k/labels/val")

In [ ]:
#Folder check 
for s in ["train", "val", "test"]:
    n = len(list((SRC_ROOT / s).glob("*.json"))) #glob finds all path names matching specific pattern and returns Generator object 
    print(s, n)

In [ ]:
#Peeking at one file

file = sorted((SRC_ROOT / "train").glob("*.json"))[0]
file_content = json.load(open(file))
print("File name :", file.name)
print("Json Keys :", list(file_content.keys()))
print("Json attributes :", file_content.get("attributes"))

In [ ]:
# Testing

path = Path("/Users/shravanthij/Documents/ML academy/driving-perception-validation/data/01_raw/bdd100k/raw_labels/train/0a0a0b1a-7c39d841.json")
with open(path) as fh:
    d = json.load(fh)
    items, weather, n_boxes, n_no_attr = [], Counter(), 0, 0

    name = d.get("name") or f.stem
    if not name.endswith(".jpg"):
        name += ".jpg"

        # per-image files store objects under frames[0]["objects"] (or "labels")
    objs = d["frames"][0].get("objects", []) 

    labels = [
        {"category": o["category"], "box2d": o["box2d"], "attributes": o.get("attributes", {})}
        for o in objs
        if o.get("box2d") is not None  # skip lanes / drivable area
        ]
    attrs = d.get("attributes") or {}
    n_no_attr += not attrs
    weather[attrs.get("weather", "missing")] += 1
    print(weather)
    n_boxes += len(labels)
    items.append({"name": name, "attributes": attrs, "labels": labels})
    print(items)


In [ ]:
# To merge all jsons into one json

def convert_split(src_dir: Path, out_path: Path):
    files = sorted(src_dir.glob("*.json"))
    assert files, f"No .json files found in {src_dir}"

    items, weather, n_boxes, n_no_attr = [], Counter(), 0, 0
    for f in files:
        with open(f) as fh:
            d = json.load(fh)

        name = d.get("name") or f.stem
        if not name.endswith(".jpg"):
            name += ".jpg"

        # per-image files store objects under frames[0]["objects"] (or "labels")
        objs = d["frames"][0].get("objects", []) if d.get("frames") else d.get("labels", [])

        labels = [
            {"category": o["category"], "box2d": o["box2d"], "attributes": o.get("attributes", {})}
            for o in objs
            if o.get("box2d") is not None  # skip lanes / drivable area
        ]
        attrs = d.get("attributes") or {}
        n_no_attr += not attrs
        weather[attrs.get("weather", "missing")] += 1
        n_boxes += len(labels)
        items.append({"name": name, "attributes": attrs, "labels": labels})

    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w") as fh:
        json.dump(items, fh)

    print(f"{src_dir.name}: {len(items)} images, {n_boxes} boxes -> {out_path}")
    print(f"  weather values: {dict(weather)}")
    if n_no_attr:
        print(f"  WARNING: {n_no_attr} files had no image-level attributes")


for split in SPLITS:
    convert_split(SRC_ROOT / split, OUT_DIR / f"bdd100k_labels_images_{split}.json")

In [ ]:
path = Path("/Users/shravanthij/Documents/ML academy/driving-perception-validation/data/01_raw/bdd100k/raw_labels/val")
convert_split(path,OUT_DIR)
SPLITS = ["val"]

# To capture count of various conditions - weather , scene , time of the day

In [ ]:
def summarize_conditions(train: list[dict], val: list[dict]) -> pd.DataFrame:
    """Count images per weather / scene / time of day for each split."""
    rows = []
    for split, items in [("train", train), ("val", val)]:
        print("split",split)
        print("items",items)
        n = len(items) # no of images
        print('n:', n)
        for key in ["weather", "scene", "timeofday"]:
            counts = Counter(x["attributes"].get(key, "missing") for x in items)
            for value, c in counts.most_common():
                print(value , " : " ,c)
                rows.append(
                    {"split": split, "attribute": key, "value": value,
                     "images": c, "pct": round(100 * c / n, 1)}
                )
    return pd.DataFrame(rows)

In [ ]:
#Test

val_path = "/Users/shravanthij/Documents/ML academy/driving-perception-validation/data/01_raw/bdd100k/labels/val1.json"
train_path = "/Users/shravanthij/Documents/ML academy/driving-perception-validation/data/01_raw/bdd100k/labels/train1.json"
train = json.load(open(train_path))
val = json.load(open(val_path))
summarize_conditions(train, val)

In [ ]:
#Test

val_path = "/Users/shravanthij/Documents/ML academy/driving-perception-validation/data/01_raw/bdd100k/labels/val1.json"

val = json.load(open(val_path))

print(type(val), len(val))   # <class 'list'> 10000
print(val[0]["attributes"])

rows = []
n = len(val)
print(val)
for key in ["weather", "scene", "timeofday"]:
    counts = Counter(x["attributes"].get(key, "missing") for x in val)
    for value, c in counts.most_common():
        print(value)
        print("c", c)
        rows.append({"attribute": key, "value": value, "images": c, "pct": round(100 * c / n, 1)})

pd.DataFrame(rows)

# Records to tables

In [ ]:
def items_to_tables(items: list[dict]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Turn image records into an images table and a boxes table."""
    imgs, boxes = [], []
    for item in items:
        print('item:' , item)
        name = item["name"]
        attrs = item.get("attributes") or {}
        print('Path:' , Path(name), Path(name).stem, Path(name).stem.split("-"))
        imgs.append(
            dict(
                name=name,
                video_id=Path(name).stem.split("-")[0],  # prefix = source video
                weather=attrs.get("weather", "undefined"),
                scene=attrs.get("scene", "undefined"),
                timeofday=attrs.get("timeofday", "undefined"),
            )
        )
        print("imgs:",imgs)
        for lab in item.get("labels") or []:
            print("lab:", lab)
            b = lab.get("box2d")
            print("box:", b)
            if b is None:
                continue
            la = lab.get("attributes") or {}
            print("trafficLightColor:", type(la.get("trafficLightColor", "none")))
            print("label attributes:",lab)
            boxes.append(
                dict(
                    name=name, category=lab["category"],
                    x1=b["x1"], y1=b["y1"], x2=b["x2"], y2=b["y2"],
                    occluded=bool(la.get("occluded", False)),
                    truncated=bool(la.get("truncated", False)),
                    traffic_light_color=la.get("trafficLightColor", "none")
                )
            )
    #box_cols = ["name", "category", "x1", "y1", "x2", "y2", "occluded", "truncated", "traffic_light_color"]
    return pd.DataFrame(imgs), pd.DataFrame(boxes)#, columns=box_cols)

In [ ]:
# Test
val_path = "/Users/shravanthij/Documents/ML academy/driving-perception-validation/data/01_raw/bdd100k/labels/val1.json"
val = json.load(open(val_path))
df1, df2 = items_to_tables(val)

In [ ]:
#Test
df1

In [ ]:
#Test
df2

In [ ]:
#Clean boxes

In [ ]:
df3=df2[["x1", "x2"]].copy()
df3[["x1", "x2"]] = [-10,1493.56]
df3

In [ ]:
df3[["x1", "x2"]].clip(0, 1280)

# Building images and boxes pool

In [ ]:
#min_box_px =2: Very small bounding boxes can represent tiny objects or annotation noise. 
#Filtering them helps avoid training on boxes that may be difficult to detect reliably.

def _build_pool(images:pd.DataFrame, boxes:pd.DataFrame, images_dir: Path, classes: List, min_box_px: float, IMG_W:int, IMG_H:int):
    b = boxes[boxes.category.isin(classes)].copy()
    print(b)
    b[["x1", "x2"]] = b[["x1", "x2"]].clip(0, IMG_W)
    b[["y1", "y2"]] = b[["y1", "y2"]].clip(0, IMG_H)
    b["w"] = b.x2 - b.x1
    b["h"] = b.y2 - b.y1
    b["area"] = b.w * b.h
    b = b[(b.w >= min_box_px) & (b.h >= min_box_px)].reset_index(drop=True) #discard bounding box whose width and height less than 2 pixels
    print('b:', b)
    print('images:',images[images.name.isin(b.name)])
    imgs = images[images.name.isin(b.name)].copy()
    exists = imgs.name.map(lambda n: (images_dir / n).exists())
    print('exists:',list(exists))
    n_missing = int((~exists).sum())
    if n_missing:
        print(f"labelled images not found in {n_missing} skipping them.", images_dir)
    imgs = imgs[exists].reset_index(drop=True)
    if imgs.empty:
        raise FileNotFoundError(
            f"No labelled images found in {images_dir}. Check data_prep.images_root in parameters."
        )
    b = b[b.name.isin(imgs.name)].reset_index(drop=True)
    imgs["stratum"] = imgs.timeofday + "|" + imgs.weather
    print(f"Pool from {images_dir}: {len(imgs)}, {len(b)} boxes")
    return imgs, b

In [ ]:
#Test
images_dir = Path("/Users/shravanthij/Documents/ML academy/driving-perception-validation/data/01_raw/bdd100k/labels/imgs")
print(images_dir)
imgs,b = _build_pool(df1, df2, images_dir, ['car', 'pedestrian', 'truck', 'traffic light', 'traffic sign', 'rider'], 2.0, 1280, 720)

In [ ]:
# When running node 3 : build_test_pool it returned 
#Pool from data/01_raw/bdd100k/images/val: 9999 images, 170741 boxes  
#To find out that 1 image below code

def items_to_tables(items: list[dict]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Turn image json records into an images table and a boxes table."""
    
    imgs, boxes = [], []
    for item in items:
        name = item["name"]
        attrs = item.get("attributes") or {}
        imgs.append(
            dict(
                name=name,
                video_id=Path(name).stem.split("-")[0],  # prefix = source video
                weather=attrs.get("weather", "undefined"),
                scene=attrs.get("scene", "undefined"),
                timeofday=attrs.get("timeofday", "undefined"),
            )
        )
        for label in item.get("labels") or []:
            box_coords = label.get("box2d")
            if box_coords is None:
                continue
            label_attrs = label.get("attributes") or {}
            boxes.append(
                dict(
                    name=name, category=label["category"],
                    x1=box_coords["x1"], y1=box_coords["y1"], x2=box_coords["x2"], y2=box_coords["y2"],
                    occluded=bool(label_attrs.get("occluded", False)),
                    truncated=bool(label_attrs.get("truncated", False)),
                    traffic_light_color=label_attrs.get("trafficLightColor", "none"),
                )
            )
    #box_cols = ["name", "category", "x1", "y1", "x2", "y2", "occluded", "truncated"]
    return pd.DataFrame(imgs), pd.DataFrame(boxes), #columns=box_cols)


import json
from pathlib import Path

# 1. Load the merged val labels (Kedro's output; use your own val.json path if different)
val = json.load(open("/Users/shravanthij/Documents/ML academy/driving-perception-validation/data/02_intermediate/bdd_labels_val.json"))

# 2. Build the two tables (needs your items_to_tables cell to have been run)
val_images_raw, val_boxes_raw = items_to_tables(val)
print(len(val_images_raw), len(val_boxes_raw))

# 3. Build the pool (needs IMG_W, IMG_H, logger and _build_pool to have been run)
classes = ["car", "pedestrian", "truck", "bus", "motorcycle", "bicycle", "rider", "traffic light", "traffic sign"]
images_dir = Path("/Users/shravanthij/Documents/ML academy/driving-perception-validation/data/01_raw/bdd100k/images/val")
pool_imgs, pool_boxes = _build_pool(val_images_raw, val_boxes_raw, images_dir, classes, 2.0, 1280, 720)

# 4. Which image was dropped, and why?
missing = set(val_images_raw.name) - set(pool_imgs.name)
print("dropped:", missing)

m = next(iter(missing))
print(val_boxes_raw[val_boxes_raw.name == m].category.value_counts())
print("file exists:", (images_dir / m).exists())
print("To check classes:" , val_boxes_raw.category.value_counts())

# Stratified sampling and leakage-aware split 


In [ ]:
def _stratified_sample(df, n, seed):
    """Sample ~n rows keeping the (timeofday|weather) mix; rare strata keep >=1."""
    rng = np.random.default_rng(seed)
    n = min(n, len(df))
    frac = n / len(df)
    parts = []
    for _, g in df.groupby("stratum"):
        k = min(len(g), max(1, int(round(len(g) * frac)))) #when rounded grouped datapoints can become 0 so at max 1 need to be taken
        parts.append(g.sample(n=k, random_state=int(rng.integers(1_000_000_000))))
    out = pd.concat(parts)
    if len(out) > n:
        out = out.sample(n=n, random_state=seed)
    return out.reset_index(drop=True)

In [ ]:
#Test
rng = np.random.default_rng(42)
random_state=int(rng.integers(1_000_000_000))
random_state

In [ ]:
#Test
rng = np.random.default_rng(42)
n=min(2000, len(pool_imgs))
frac = n/len(pool_imgs)
frac
parts=[]
for _, g in pool_imgs.groupby("stratum"):
    #print(_,":",min(len(g),max(1, int(round(len(g)*frac)))))
    k=min(len(g),max(1, int(round(len(g)*frac))))
    parts.append(g.sample(n=k,random_state=int(rng.integers(1_000_000_000))))
out = pd.concat(parts)   
print(out)
if len(out) > n:
    out = out.sample(n=n, random_state=42)
out.reset_index(drop=True)

In [ ]:
#Test
pool_imgs[pool_imgs['stratum']=='dawn/dusk|clear']

In [ ]:
pool_imgs

In [ ]:
pool_boxes

In [ ]:
def _group_split(df, val_frac, seed):
    """Split by video id within each stratum so no video spans train and val."""
    rng = np.random.default_rng(seed)
    val_videos = set()
    for _, g in df.groupby("stratum"):
        vids = g.video_id.unique()
        rng.shuffle(vids)
        k = int(round(len(vids) * val_frac))
        val_videos.update(vids[:k])
    is_val = df.video_id.isin(val_videos)
    return df[~is_val].reset_index(drop=True), df[is_val].reset_index(drop=True)

In [ ]:
#Test
rng = np.random.default_rng(42)
val_frac = 0.20
val_videos = set()
for _, g in pool_imgs.groupby("stratum"):
    vids = g.video_id.unique()
    rng.shuffle(vids)
    #print(vids)
    k=int(round(len(vids) * val_frac))
    s = set(vids[:k])
    #print("already present:",len(s),s.intersection(val_videos))
    val_videos.update(vids[:k])

is_val = pool_imgs.video_id.isin(val_videos)
print(~is_val)  
    

In [ ]:
#Test
pool_imgs.groupby("video_id")["stratum"].nunique().value_counts()

In [ ]:
def _check_no_leakage(splits: dict):
    print("input: ",splits)
    names = list(splits)
    print("names: ", names)
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            overlap = set(splits[names[i]].video_id) & set(splits[names[j]].video_id)
            assert not overlap, f"Video leakage between {names[i]} and {names[j]}: {len(overlap)} videos"
    print("Leakage check passed: no video id appears in more than one split.")

In [ ]:
def split_dataset(train_imgs, train_boxes, test_imgs, test_boxes, params: dict):
    """Build train / val / test manifests (images and boxes, with a `split` column)."""
    n_train, n_val, n_test, seed = (params[k] for k in ("n_train", "n_val", "n_test", "seed"))
 
    sampled = _stratified_sample(train_imgs, n_train + n_val, seed)
    tr, va = _group_split(sampled, n_val / (n_train + n_val), seed)
    te = _stratified_sample(test_imgs, n_test, seed + 1)
 
    # (images, boxes pool, official image folder the files live in)
    splits = {
        "train": (tr, train_boxes, "train"),
        "val": (va, train_boxes, "train"),
        "test": (te, test_boxes, "val"),
    }
    #_check_no_leakage({k: v[0] for k, v in splits.items()})
 
    img_parts, box_parts = [], []
    for split, (imgs, pool_boxes, src_dir) in splits.items():
        b = pool_boxes[pool_boxes.name.isin(imgs.name)]
        counts = b.groupby("name").size()
        img_parts.append(imgs.assign(
            split=split, source_dir=src_dir,
            n_boxes=imgs.name.map(counts).fillna(0).astype(int)))
        box_parts.append(b.assign(split=split))
        #logger.info("%s: %d images, %d boxes", split, len(imgs), len(b))
 
    return pd.concat(img_parts, ignore_index=True), pd.concat(box_parts, ignore_index=True)

In [ ]:
#Test
train_imgs=imgs
test_imgs = imgs
train_boxes=b
test_boxes =b
params = {"n_train": 40,
  "n_val": 15,
  "n_test": 30,
  "seed": 42}
split_img, split_box = split_dataset(train_imgs, train_boxes, test_imgs, test_boxes, params)

In [ ]:
split_box

In [ ]:
#check no leakage
splits = {"n_train": [1,2,3,4,5,6,7], "n_val":[2,8,9,4,5,6]}
names = list(params)
overlap = set(splits[names[0]]) & set(splits[names[1]])
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        overlap = set(splits[names[0]]) & set(splits[names[1]])
        assert not overlap, f"Video leakage between {names[i]} and {names[j]}: {len(overlap)} videos"


In [ ]:
imgs

# Write the YOLO dataset

In [ ]:
def write_yolo_dataset(images_manifest, boxes_manifest, params: dict) -> str:
    """Copy (or symlink) images and write YOLO labels + dataset.yaml. Returns the yaml path."""
    out = Path(params["yolo_out_dir"])
    classes = params["classes"]
    class_to_id = {c: i for i, c in enumerate(classes)}
    images_root = Path(params["images_root"])
 
    # Start clean so reruns with a different sample never leave stale files behind.
    for sub in ("images", "labels"):
        shutil.rmtree(out / sub, ignore_errors=True)
    for split in ("train", "val", "test"):
        (out / "images" / split).mkdir(parents=True, exist_ok=True)
        (out / "labels" / split).mkdir(parents=True, exist_ok=True)
 
    by_img = {k: g for k, g in boxes_manifest.groupby(["split", "name"])}
    for row in images_manifest.itertuples():
        src = (images_root / row.source_dir / row.name).resolve()
        dst = out / "images" / row.split / row.name
        if params["link_images"]:
            dst.symlink_to(src)
        else:
            shutil.copy2(src, dst)
 
        g = by_img[(row.split, row.name)]
        cx = ((g.x1 + g.x2) / 2 / IMG_W).round(6)
        cy = ((g.y1 + g.y2) / 2 / IMG_H).round(6)
        w = (g.w / IMG_W).round(6)
        h = (g.h / IMG_H).round(6)
        cls = g.category.map(class_to_id)
        lines = [f"{c} {x} {y} {ww} {hh}" for c, x, y, ww, hh in zip(cls, cx, cy, w, h)]
        (out / "labels" / row.split / f"{Path(row.name).stem}.txt").write_text("\n".join(lines) + "\n")
 
    yaml_lines = [
        f"path: {out.resolve()}", "train: images/train", "val: images/val", "test: images/test", "names:",
    ] + [f"  {i}: {c}" for i, c in enumerate(classes)]
    yaml_path = out / "dataset.yaml"
    yaml_path.write_text("\n".join(yaml_lines) + "\n")
    logger.info("YOLO dataset written to %s", out.resolve())
    return str(yaml_path)

In [ ]:
#Test
classes = ["car", "person", "truck", "bus", "motor", "bike", "rider", "traffic light", "traffic sign"]
class_to_id = {c: i for i, c in enumerate(classes)}
class_to_id
out=Path("data/01_raw/bdd100k/images/label")
yaml_lines = [
        f"path: {out.resolve()}", "train: images/train", "val: images/val", "test: images/test", "names:",
    ] + [f"  {i}: {c}" for i, c in enumerate(classes)]
print(yaml_lines)

In [ ]:
images_dir: data/01_raw/bdd100k/images/label

In [ ]:
by_img = {k: g for k, g in split_box.groupby(["split", "name"])}


In [ ]:
for row in split_img.itertuples():
    print(Path(row.name).stem)
    print(by_img[(row.split, row.name)])
    g = by_img[(row.split, row.name)]
    cx = ((g.x1 + g.x2) / 2 / IMG_W).round(6)
    cy = ((g.y1 + g.y2) / 2 / IMG_H).round(6)
    w = (g.w / IMG_W).round(6)
    h = (g.h / IMG_H).round(6)
    cls = g.category.map(class_to_id)

In [ ]:
split_box

In [ ]:
split_img

# Data report

In [ ]:
def make_data_report(images_manifest, boxes_manifest, params: dict) -> str:
    """Markdown data report: size, class balance, conditions, small boxes, occlusion."""
    classes = params["classes"]
    img_tbl, box_tbl = images_manifest, boxes_manifest
    lines = ["# BDD100K subset: data report\n"]
 
    n_imgs = img_tbl.groupby("split").size().rename("images")
    n_boxes = box_tbl.groupby("split").size().rename("boxes")
    lines += ["## Size per split", "```", pd.concat([n_imgs, n_boxes], axis=1).to_string(), "```\n"]
 
    cls_counts = box_tbl.pivot_table(index="category", columns="split", values="name",
                                     aggfunc="count", fill_value=0).reindex(classes)
    cls_pct = (cls_counts / cls_counts.sum() * 100).round(1)
    lines += ["## Class counts (boxes)", "```", cls_counts.to_string(), "```",
              "## Class share per split (% of boxes)", "```", cls_pct.to_string(), "```\n"]
 
    for col in ["timeofday", "weather", "scene"]:
        t = pd.crosstab(img_tbl[col], img_tbl["split"], normalize="columns").mul(100).round(1)
        counts = pd.crosstab(img_tbl[col], img_tbl["split"])
        lines += [f"## {col}: % of images per split", "```", t.to_string(), "```",
                  f"## {col}: image counts per split", "```", counts.to_string(), "```\n"]
 
    small = (box_tbl.area < 32 * 32).groupby([box_tbl.split, box_tbl.category]).mean()
    small = small.mul(100).round(1).unstack(0).reindex(classes)
    occ = box_tbl.groupby("category")[["occluded", "truncated"]].mean().mul(100).round(1).reindex(classes)
    lines += ["## Small boxes (<32x32 px), % per class and split", "```", small.to_string(), "```",
              "## Occluded / truncated boxes, % per class", "```", occ.to_string(), "```\n"]
 
    return "\n".join(lines)

In [ ]:
#Test
n_imgs = split_img.groupby("split").size().rename("images")
n_boxes = split_box.groupby("split").size().rename("boxes")
pd.concat([n_imgs, n_boxes], axis=1)

In [ ]:
cls_counts=split_box.pivot_table(index="category", columns="split", values="name", aggfunc="count", fill_value=0).reindex(classes).fillna(0)

In [ ]:
cls_counts

In [ ]:
cls_counts.sum()

In [ ]:
(cls_counts / cls_counts.sum() * 100).round(1)

In [ ]:
for col in ["timeofday", "weather", "scene"]:
    print(pd.crosstab(split_img[col], split_img["split"], normalize="columns").mul(100).round(1))
    print("---------")
    print(pd.crosstab(split_img[col], split_img["split"]))

In [ ]:
split_img

In [ ]:
small = (split_box.area < 32 * 32).groupby([split_box.split, split_box.category]).mean()
small.mul(100).round(1).unstack(0).reindex(classes, fill_value=0)

In [ ]:
split_box

In [ ]:
split_box.groupby("category")[["occluded", "truncated"]].mean().mul(100).round(1).reindex(classes)